# Parameter estimation: TINN vs PyDDM (and BayesFlow), local run

Runs the simulation benchmark for one model at a time: test datasets, PyDDM, TINN, optional BayesFlow,
then the tables and black-and-white PDF figures. Results go to `results/<MODEL>/` next to this notebook,
tables and figures to `results/<MODEL>/paper/`.

In the paper, PyDDM and TINN were run with this notebook and BayesFlow with `bayesflow_colab.ipynb`
(GPU), on the same `test_datasets.npz`.

**Resuming.** Every step continues where it stopped: datasets already in a part file are skipped.
To start a model from scratch, delete `results/<MODEL>/`.

## 1. Settings

In [ ]:
import os
MODEL = "ddm"                # ddm | urgency | ou | dmc | collapse (paper); ddm3 | urgency3 | ou3 | collapse3 (3 bound conditions)
N_DATA = 200             # number of test datasets
N_TRIALS = 1000         # trials per dataset, split evenly over the conditions (dmc: 2; urgency3 / ou3: 3,
                        # so 3000 gives 1000 per condition)
WORKERS = max(1, (os.cpu_count() or 2) - 1)   # parallel processes for PyDDM and TINN

RUN_PYDDM, RUN_TINN, RUN_BAYESFLOW = True, True, False
TINN_WARM, TINN_ITERS = 2000, 6000      # TINN iterations (network warm-up, then joint fit)
TINN_DEVICE = "auto"    # "auto" = GPU if PyTorch sees one (CUDA build of torch), else CPU; or "cuda" / "cpu"
TINN_WORKERS = None     # parallel TINN processes; None = WORKERS on CPU, 3 on one GPU
BF_SIM, BF_EPOCHS = 20000, 50           # BayesFlow training datasets and epochs (slow on a CPU)
print(MODEL, "with", WORKERS, "parallel processes")

## 2. Results folder and helpers

In [ ]:
import os, sys, time, glob, subprocess
import numpy as np, pandas as pd

CMP = os.path.abspath(".")                          # this folder (2_parameter_estimation)
assert os.path.exists(os.path.join(CMP, "fit_pinn.py")), "open this notebook from 2_parameter_estimation/"
sys.path.insert(0, CMP)
R = os.path.abspath(os.path.join("results", MODEL))   # results next to this notebook
os.makedirs(R, exist_ok=True)
DATA = os.path.join(R, "test_datasets.npz")
ENV = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
print("code:", CMP)
print("results:", R)


def n_rows(pattern):
    return sum(len(pd.read_csv(f)) for f in glob.glob(pattern) if os.path.getsize(f) > 0)


def run_script(script, *args):
    """runs one of the comparison scripts in its own Python process and shows its output."""
    p = subprocess.Popen([sys.executable, "-u", os.path.join(CMP, script), *map(str, args)], cwd=CMP,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for l in p.stdout:                       # shown as it comes, so long steps show progress
        if not l.startswith("INFO") and "Warning" not in l:
            print(l, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"{script} failed")


def run_parallel(script, tag, extra=()):
    """runs `script` on WORKERS slices of the datasets at once, shows progress, merges the parts.
    Resumable: datasets already in a part file are skipped, so after a stop just run the cell again."""
    step = -(-N_DATA // WORKERS)
    procs = []
    for w in range(WORKERS):
        first, last = w * step, min((w + 1) * step, N_DATA)
        if first >= last:
            continue
        log = open(os.path.join(R, f"log_{tag}_{w}.txt"), "a")
        procs.append(subprocess.Popen(
            [sys.executable, os.path.join(CMP, script), "--data", DATA, "--first", str(first),
             "--last", str(last), "--out", os.path.join(R, f"{tag}_part{w}.csv"), *map(str, extra)],
            stdout=log, stderr=subprocess.STDOUT, env=ENV, cwd=CMP))
    t = time.time()
    try:
        while any(p.poll() is None for p in procs):
            time.sleep(10)
            print(f"\r{tag}: {n_rows(os.path.join(R, tag + '_part*.csv'))}/{N_DATA} datasets, "
                  f"{(time.time() - t) / 60:.1f} min", end="", flush=True)
    except KeyboardInterrupt:          # the stop button of the notebook also stops the workers
        for p in procs:
            p.terminate()
        raise
    print()
    if any(p.returncode for p in procs):
        for f in sorted(glob.glob(os.path.join(R, f"log_{tag}_*.txt"))):
            print(f, "\n", open(f).read()[-2000:])
        raise RuntimeError(f"{tag} failed, see the logs above")
    df = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(R, f"{tag}_part*.csv")))])
    df = df.drop_duplicates("dataset").sort_values("dataset")
    df.to_csv(os.path.join(R, f"{tag}.csv"), index=False)
    print(f"{tag}: {len(df)} datasets -> {os.path.join(R, tag + '.csv')}")
    return df

## 3. Test datasets

Parameters are drawn uniformly from `MODELS[MODEL]["ranges"]` in `common.py`; the same ranges are the BayesFlow prior.

In [ ]:
from common import make_test_datasets, load_test_datasets, ranges_of

if not os.path.exists(DATA):
    make_test_datasets(MODEL, N_DATA, N_TRIALS, path=DATA)
d = load_test_datasets(DATA)
assert d["model"] == MODEL, f"{DATA} belongs to {d['model']}"
N_DATA = len(d["rt"])
print(MODEL, "ranges:", ranges_of(MODEL))
print(f"{d['rt'].shape[0]} datasets x {d['rt'].shape[1]} trials;  "
      f"P(upper) {d['choice'].mean(1).min():.2f}-{d['choice'].mean(1).max():.2f};  "
      f"mean RT {np.nanmean(d['rt'], 1).min():.2f}-{np.nanmean(d['rt'], 1).max():.2f} s")

## 4. PyDDM

Maximum likelihood with differential evolution; the default 2% mixture is switched off. For the models other than the DDM the robust likelihood is used, because PyDDM truncates the density tail.

In [ ]:
if RUN_PYDDM:
    pyddm_df = run_parallel("fit_pyddm.py", "pyddm")

## 5. TINN

Runs on the GPU when PyTorch sees one (`TINN_DEVICE`); several processes (default 3) share one GPU. On a CPU it takes about 4 to 5 minutes per dataset on one core.

In [ ]:
if RUN_TINN:
    tinn_df = run_parallel("fit_pinn.py", "tinn", ["--warm", TINN_WARM, "--iters", TINN_ITERS])

## 6. BayesFlow (optional, local)

Runs in its own process, because BayesFlow with the torch backend switches gradients off for the whole process. Slow on a CPU. The BayesFlow results in the paper come from `bayesflow_colab.ipynb`.

In [ ]:
if RUN_BAYESFLOW and not os.path.exists(os.path.join(R, "bayesflow.csv")):
    run_script("fit_bayesflow.py", "--data", DATA, "--out", os.path.join(R, "bayesflow.csv"),
               "--n-sim", BF_SIM, "--epochs", BF_EPOCHS, "--diag-dir", os.path.join(R, "paper"))

## 7. Tables and figures

Every method whose result file exists is included. Output: `results/<MODEL>/paper/`.

In [ ]:
run_script("make_figures.py", "--results", R)

from IPython.display import Image, display
for f in sorted(glob.glob(os.path.join(R, "paper", "fig*.png"))):
    print(os.path.basename(f))
    display(Image(f, width=900))

## 8. Summary table over all models

Collects every model with results in `results/` into one LaTeX table (`results/summary_table.tex`).

In [ ]:
run_script("make_summary_table.py", "--results", os.path.abspath("results"))